# 1단계: Korean sLLM 일반 텍스트 사전학습 (Colab)

기존 `ModelConfig`(24L/d768, vocab 32768, MTP 16)를 그대로 사용해 랜덤 초기화부터 학습합니다.
기존 SentencePiece 토크나이저를 사용하며 이후 SFT에서도 동일 파일을 사용해야 합니다.

1. Colab에서 GPU 런타임을 선택하고 아래 셀을 순서대로 실행하세요.
   저장소의 `train_pretrain.py`와 `pretrain_data.py`를 그대로 사용합니다.
2. 데이터 폴더는 `/content/drive/MyDrive/korean_sllm_data/pretrain`입니다.
   Drive의 `korean_sllm_data/pretrain/pretrain_train.json`, `pretrain_val.json`에 일반 텍스트를 준비하세요.
   각 JSON 대신 같은 이름의 `.tar.xz` 압축 파일을 업로드해도 자동 해제합니다.
   파일은 `["한국어 문서 본문...", "다음 문서..."]` 형식의 JSON 문자열 배열입니다. 채팅 템플릿이나 user/assistant 필드는 사용하지 않습니다.
   JSON 문자열의 줄바꿈은 `\n`으로 이스케이프하세요. val은 문서 단위로 분리하고 train 중복을 제거하세요.
   코퍼스는 사용자가 준비합니다. 아래 예시 문서는 형식 설명용이며 학습 데이터로 자동 사용하지 않습니다.
   체크포인트(`best.pt`, `last.pt`)와 토크나이저(`spm.model`)도 같은 `pretrain` 폴더에 저장합니다.
3. 기본 목표는 **총 5epoch**, 한 번 실행할 분량은 **0.25epoch**입니다(약 20회).
   다음 날 같은 Drive 폴더로 이 노트북을 처음부터 실행하면 `last.pt`에서 자동 재개합니다.
   `TOTAL_EPOCHS=5`는 매번 유지하고 `SESSION_EPOCHS=0.25`만 실행 분량으로 사용합니다.
   학습률은 전체 5epoch 기준으로 이어지며 warmup을 다시 시작하지 않습니다.
   모델·옵티마이저·난수 상태와 셔플된 데이터의 다음 배치 위치를 복원합니다.
   500스텝마다, 그리고 세션 종료 시 저장합니다. 런타임이 갑자기 끊기면 마지막 저장 이후 작업은 다시 수행합니다.
   재개 중에는 데이터·토크나이저·batch/accum·전체 스케줄을 유지하세요.
   이전 노트북 형식의 체크포인트에는 정확한 재개 정보가 없어 새 출력 폴더가 필요합니다.
   세션 분량은 optimizer step 단위로 올림하므로 정확히 0.25epoch에서 조금 벗어날 수 있습니다.
   Colab 로컬 캐시는 새 런타임에서 다시 생성하므로 데이터 복사·토큰화 시간이 별도로 듭니다.
4. 총 목표 학습이 끝나면 `colab_train_think_weight_01.ipynb`로 인스트럭션 학습을 진행하세요.

문서마다 BOS/EOS를 붙이고 모든 다음 토큰에 weight=1로 main CE와 MTP CE를 계산합니다.
긴 문서를 길이 제한으로 버리지 않고 이어 붙여 패킹합니다. 문서 간 attention은 허용하며 EOS로 경계를 표시합니다.
윈도우는 2048토큰, 겹침은 256토큰(이동 간격 1792)입니다.
마지막 윈도우는 코퍼스 끝에 맞춰 겹침을 늘려 남는 토큰도 포함합니다.
전체 코퍼스가 2048토큰보다 짧으면 그 길이 그대로 사용하며 패딩하지 않습니다.
마지막 작은 배치도 학습하고 epoch 스텝은 올림 계산하여 전체 데이터가 포함됩니다. MTP는 각 윈도우 안에서만 예측합니다.
캐시는 원문+토크나이저 해시로 구분하고 문서 단위로 디스크에 기록합니다(전체 코퍼스를 RAM에 올리지 않음).

기본값은 기존 노트북과 같은 **96GB GPU / seq 2048 / batch 8 / accum 4** 기준입니다.
T4 등에서는 batch를 낮추고 `GRAD_CHECKPOINTING=True`로 조정해야 하며, 메모리 적합성은 실행해서 확인해야 합니다.
사전학습에는 충분한 양의 중복 제거된 한국어 코퍼스가 필요합니다. 총 epochs=5, 세션 epochs=0.25, lr=3e-4는 시작값이며 성능 개선을 보장하지 않습니다.


In [ ]:
# 1) 리포 준비 (Colab GPU 런타임을 먼저 선택하세요)
from pathlib import Path
import subprocess
import os
import sys
REPO_URL = "https://github.com/MinsuChae/korean_sllm.git"
REPO_DIR = Path('/content/korean_sllm')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
os.chdir(REPO_DIR)
# 저장소에 커밋된 사전학습 코드를 그대로 실행합니다.
assert Path('train_pretrain.py').is_file(), '사전학습 스크립트가 포함된 저장소 버전을 사용하세요.'
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt'], check=True)


In [ ]:
# 2) Drive 및 데이터/출력 경로
from google.colab import drive
import shutil
import hashlib
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Colab 런타임 유형을 GPU로 변경하세요.'
DATA_DIR = Path('/content/drive/MyDrive/korean_sllm_data/pretrain')
CKPT_DIR = DATA_DIR
CKPT_DIR.mkdir(parents=True, exist_ok=True)
# 다음 날에도 같은 폴더로 실행하면 last.pt에서 자동 재개합니다.
RESUME = str(CKPT_DIR / 'last.pt') if (CKPT_DIR / 'last.pt').is_file() else None
if RESUME:
    assert (CKPT_DIR / 'spm.model').is_file()
    shutil.copy2(CKPT_DIR / 'spm.model', 'tokenizer/spm.model')
else:
    assert not any(CKPT_DIR.glob('*.pt')), '기존 실행이 있습니다. RESUME을 지정하거나 새 CKPT_DIR을 사용하세요.'
    # Drive에 업로드된 토크나이저가 있으면 새 학습에서도 사용합니다.
    if (CKPT_DIR / 'spm.model').is_file():
        shutil.copy2(CKPT_DIR / 'spm.model', 'tokenizer/spm.model')
    else:
        shutil.copy2('tokenizer/spm.model', CKPT_DIR / 'spm.model')
print('GPU:', torch.cuda.get_device_name())
print('tokenizer SHA256:', hashlib.sha256(Path('tokenizer/spm.model').read_bytes()).hexdigest())


In [ ]:
# 3) Drive 코퍼스를 Colab 로컬 디스크에 복사 (반복 학습 I/O 개선)
from pretrain_data import iter_texts
import tarfile
LOCAL_DATA = Path('/content/pretrain_data')
LOCAL_DATA.mkdir(parents=True, exist_ok=True)
for split in ('train', 'val'):
    source = DATA_DIR / f'pretrain_{split}.json'
    local = LOCAL_DATA / source.name
    if source.is_file():
        shutil.copy2(source, local)
    else:
        archive = source.with_name(source.name + '.tar.xz')
        if not archive.is_file():
            raise FileNotFoundError(f'{source} 또는 {archive}를 Drive에 업로드하세요.')
        print('압축 해제:', archive, flush=True)
        with tarfile.open(archive, 'r:xz') as tar:
            matches = [m for m in tar.getmembers() if m.isfile() and Path(m.name).name == source.name]
            if len(matches) != 1:
                raise ValueError(f'{archive}: {source.name} 파일이 정확히 하나 있어야 합니다.')
            with tar.extractfile(matches[0]) as src, local.open('wb') as dst:
                shutil.copyfileobj(src, dst)
    documents = iter_texts(local)
    try:
        assert next(documents, None) is not None, f'{source}: 비어 있는 코퍼스'
    finally:
        documents.close()
    print(split, 'bytes:', local.stat().st_size)


In [ ]:
# 4) 사전학습. 모델 구조는 변경하지 않습니다.
SEQ_LEN = 2048
OVERLAP = 256
BATCH_SIZE = 8
GRAD_ACCUM = 4
GRAD_CHECKPOINTING = False
TOTAL_EPOCHS = 5  # 전체 목표: 재개할 때도 5로 유지
SESSION_EPOCHS = 0.25  # 이번 실행에서 추가 학습할 분량
cmd = [
    sys.executable, 'train_pretrain.py', '--stage', 'pretrain',
    '--data-dir', str(LOCAL_DATA), '--cache-dir', '/content/pretrain_cache',
    '--ckpt-dir', str(CKPT_DIR), '--seq-len', str(SEQ_LEN),
    '--overlap', str(OVERLAP),
    '--batch-size', str(BATCH_SIZE), '--grad-accum', str(GRAD_ACCUM),
    '--epochs', str(TOTAL_EPOCHS), '--session-epochs', str(SESSION_EPOCHS), '--lr', '3e-4', '--warmup-steps', '500',
    '--eval-every', '250', '--save-every', '500',
]
if GRAD_CHECKPOINTING:
    cmd.append('--grad-checkpointing')
# 같은 런타임에서 이 셀만 다시 실행해도 최신 체크포인트를 사용합니다.
resume_path = CKPT_DIR / 'last.pt'
if resume_path.is_file():
    cmd += ['--resume', str(resume_path)]
subprocess.run(cmd, check=True)


In [ ]:
# 5) SFT로 전달할 파일 확인 및 Drive flush
# best.pt: 검증 main loss 최저 가중치 / last.pt: 사전학습 재개용
for filename in ('best.pt', 'last.pt', 'spm.model'):
    path = CKPT_DIR / filename
    assert path.is_file(), f'산출물 누락: {path}'
    print(path, path.stat().st_size, 'bytes')
checkpoint = torch.load(CKPT_DIR / 'last.pt', map_location='cpu', weights_only=True)
progress = checkpoint['step'] / checkpoint['run_config']['steps_per_epoch']
finished = checkpoint['step'] >= checkpoint['run_config']['total_steps']
print(f'누적 학습: {progress:.4f} / {TOTAL_EPOCHS} epochs')
del checkpoint
drive.flush_and_unmount()
if finished:
    print('목표 완료. colab_train_think_weight_01.ipynb에서 pretrain/best.pt로 SFT 시작')
else:
    print('세션 저장 완료. 다음 실행에서 이 노트북을 처음부터 실행하면 자동으로 이어집니다.')
